In [5]:
"""
Synthetic microbiome dataset generator
========================================
Metagenomics Summer School - ML Decision Game

Produces a students-facing CSV (counts + health_status + batch) and an
instructor-only answer key (which taxa are the TRUE biological signal,
which taxa are the TRUE batch artifact, and how strongly batch is
confounded with health_status).

Design choices (deliberate, for the decision-game to work):
- N_SAMPLES = 150, N_TAXA = 400 (features >> samples, realistic).
- Baseline community: log-normal abundances -> few dominant taxa, long
  tail of rare taxa -> realistic sparsity/zero-inflation after multinomial
  sampling.
- CLASS_PROBS controls class imbalance. For example, p=[0.7, 0.3]
  gives approximately 70% Control and 30% Case if the labels are ordered
  as ["Control", "Case"]. This allows students to see why accuracy can
  be misleading and why balanced accuracy, F1-score, ROC/PR curves, and
  the confusion matrix should be inspected together.
- SIGNAL_TAXA: subset of taxa shifted by health_status. Effect size is
  MODEST and spread over many taxa (biological signal is usually subtle
  and diffuse, not a single dramatic biomarker).
- BATCH_TAXA: a disjoint subset of taxa shifted by batch. Effect size is
  LARGER than the biological one, on purpose: this is what makes naive
  PCA / clustering (unsupervised, blind to labels) latch onto batch
  structure instead of health status, while a supervised model (which
  is explicitly optimized against the label) can still recover a modest
  but real signal. This is what you want students to *discover*, not be
  told.
- CONFOUND_STRENGTH controls how much batch assignment correlates with
  health_status (0 = independent/balanced, 1 = perfectly confounded).
  Kept moderate here so grouped CV vs random CV gives an interesting,
  not catastrophic, gap.
- Sequencing depth varies per sample (log-normal), independent of both
  health_status and batch, and is not informative by design.
"""

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# Parameters (feel free to tune for different editions of the course)
# ----------------------------------------------------------------------
RANDOM_STATE = 42

N_SAMPLES = 150
N_TAXA = 400

N_SIGNAL_TAXA = 20          # taxa truly associated with health_status
N_BATCH_TAXA = 20           # taxa truly associated with batch (disjoint from signal)
SIGNAL_EFFECT_SIZE = 0.4    # log-fold shift for signal taxa (modest)
BATCH_EFFECT_SIZE = 1.5     # log-fold shift for batch taxa (larger than signal)

N_BATCHES = 3
CONFOUND_STRENGTH = 0.7     # 0 = batch independent of status, 1 = fully confounded

DEPTH_LOGMEAN = 10.5        # ~ mean sequencing depth exp(10.5) ~ 36,000 reads
DEPTH_LOGSD = 0.35

DIRICHLET_NOISE = 0.15      # per-sample biological noise on top of group means


CLASS_LABELS = ["Case", "Control"]
CLASS_PROBS = [0.5, 0.5]

OUT_STUDENT_CSV = "microbiome_synthetic.csv"
OUT_INSTRUCTOR_KEY = "instructor_answer_key.csv"

rng = np.random.default_rng(RANDOM_STATE)

# ----------------------------------------------------------------------
# 1. Baseline community composition (shared "world")
# ----------------------------------------------------------------------
taxa_names = [f"Taxon_{i:04d}" for i in range(1, N_TAXA + 1)]

# Core/rare mixture baseline -> a set of moderately abundant "core" taxa
# plus a long tail of rare taxa. This gives realistic zero-inflation
# (~60-70% zeros after multinomial sequencing) without any single taxon
# implausibly dominating the whole community.
N_CORE_TAXA = 40
core_log_abund = rng.normal(loc=2.0, scale=1.0, size=N_CORE_TAXA)
rare_log_abund = rng.normal(loc=-6.0, scale=1.5, size=N_TAXA - N_CORE_TAXA)
baseline_log_abundance = np.concatenate([core_log_abund, rare_log_abund])
rng.shuffle(baseline_log_abundance)  # randomize which taxon is core vs rare

# ----------------------------------------------------------------------
# 2. Assign ground-truth signal and batch taxa (disjoint sets)
# ----------------------------------------------------------------------
all_idx = rng.permutation(N_TAXA)
signal_taxa_idx = all_idx[:N_SIGNAL_TAXA]
batch_taxa_idx = all_idx[N_SIGNAL_TAXA:N_SIGNAL_TAXA + N_BATCH_TAXA]

# Direction of effect per taxon (up or down), fixed once
signal_direction = rng.choice([-1, 1], size=N_SIGNAL_TAXA)
batch_direction = rng.choice([-1, 1], size=(N_BATCHES, N_BATCH_TAXA))

# ----------------------------------------------------------------------
# 3. Sample-level metadata: health_status and (confounded) batch
# ----------------------------------------------------------------------
health_status = rng.choice(CLASS_LABELS, size=N_SAMPLES, p=CLASS_PROBS)

batch_labels = [f"batch_{b+1}" for b in range(N_BATCHES)]
batch = np.empty(N_SAMPLES, dtype=object)

for i, status in enumerate(health_status):
    if rng.random() < CONFOUND_STRENGTH:
        # confounded draw: skewed towards one "preferred" batch per status
        preferred = 0 if status == "Control" else N_BATCHES - 1
        probs = np.full(N_BATCHES, (1 - 0.6) / (N_BATCHES - 1))
        probs[preferred] = 0.6
        batch[i] = rng.choice(batch_labels, p=probs)
    else:
        # independent, uniform draw
        batch[i] = rng.choice(batch_labels)

sequencing_depth = rng.lognormal(mean=DEPTH_LOGMEAN, sigma=DEPTH_LOGSD, size=N_SAMPLES).astype(int)

# ----------------------------------------------------------------------
# 4. Build per-sample true log-abundance profile, sample counts
# ----------------------------------------------------------------------
counts = np.zeros((N_SAMPLES, N_TAXA), dtype=int)

for i in range(N_SAMPLES):
    log_abund = baseline_log_abundance.copy()

    # biological (health_status) effect
    sign = 1 if health_status[i] == "Case" else -1
    log_abund[signal_taxa_idx] += sign * SIGNAL_EFFECT_SIZE * signal_direction

    # technical (batch) effect
    b_idx = batch_labels.index(batch[i])
    log_abund[batch_taxa_idx] += BATCH_EFFECT_SIZE * batch_direction[b_idx]

    # per-sample biological noise
    log_abund += rng.normal(loc=0.0, scale=DIRICHLET_NOISE, size=N_TAXA)

    # true relative composition for this sample
    abund = np.exp(log_abund)
    proportions = abund / abund.sum()

    # multinomial sequencing -> realistic sparsity / zero-inflation
    counts[i, :] = rng.multinomial(sequencing_depth[i], proportions)

# ----------------------------------------------------------------------
# 5. Assemble student-facing dataframe
# ----------------------------------------------------------------------
df = pd.DataFrame(counts, columns=taxa_names)
df.insert(0, "sample_id", [f"S{i+1:03d}" for i in range(N_SAMPLES)])
df["health_status"] = health_status
df["batch"] = batch
df = df.set_index("sample_id")

df.to_csv(OUT_STUDENT_CSV)

# ----------------------------------------------------------------------
# 6. Instructor-only answer key (do NOT distribute to students)
# ----------------------------------------------------------------------
key_rows = []
for idx, d in zip(signal_taxa_idx, signal_direction):
    key_rows.append({"taxon": taxa_names[idx], "true_role": "health_signal",
                      "direction": "up_in_Case" if d == 1 else "down_in_Case",
                      "effect_size": SIGNAL_EFFECT_SIZE})
for idx in batch_taxa_idx:
    key_rows.append({"taxon": taxa_names[idx], "true_role": "batch_artifact",
                      "direction": "varies_by_batch", "effect_size": BATCH_EFFECT_SIZE})

key_df = pd.DataFrame(key_rows)
key_df.to_csv(OUT_INSTRUCTOR_KEY, index=False)

# ----------------------------------------------------------------------
# 7. Sanity checks / summary (printed for you, the instructor)
# ----------------------------------------------------------------------
print("Student CSV:", OUT_STUDENT_CSV, "-> shape:", df.shape)
print("Instructor key:", OUT_INSTRUCTOR_KEY, "-> shape:", key_df.shape)
print()

# Basic checks
assert np.isclose(np.sum(CLASS_PROBS), 1.0), "CLASS_PROBS must sum to 1."
assert len(set(signal_taxa_idx).intersection(set(batch_taxa_idx))) == 0, \
    "Signal taxa and batch taxa must be disjoint."

print("Class balance:")
class_counts = df["health_status"].value_counts().reindex(CLASS_LABELS)
class_props = df["health_status"].value_counts(normalize=True).reindex(CLASS_LABELS)

print(class_counts)
print()
print("Class proportions:")
print(class_props.round(3))

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Imbalance ratio majority/minority: {imbalance_ratio:.2f}")
print()

print("Batch balance:")
print(df["batch"].value_counts().sort_index())
print()

print("Confounding check (status x batch):")
ct = pd.crosstab(df["health_status"], df["batch"]).reindex(CLASS_LABELS)
print(ct)
print()

print("Confounding check, row-normalized:")
ct_norm = pd.crosstab(
    df["health_status"],
    df["batch"],
    normalize="index",
).reindex(CLASS_LABELS)
print(ct_norm.round(3))
print()

# Sequencing depth checks
df_depth = df[taxa_names].sum(axis=1)

print("Sequencing depth by health_status:")
print(
    pd.DataFrame({
        "median_depth": df_depth.groupby(df["health_status"]).median(),
        "mean_depth": df_depth.groupby(df["health_status"]).mean(),
    }).reindex(CLASS_LABELS).round(1)
)
print()

print("Sequencing depth by batch:")
print(
    pd.DataFrame({
        "median_depth": df_depth.groupby(df["batch"]).median(),
        "mean_depth": df_depth.groupby(df["batch"]).mean(),
    }).sort_index().round(1)
)
print()

zero_fraction = (df[taxa_names].values == 0).mean()
sample_zero_fraction = (df[taxa_names] == 0).mean(axis=1)

print(f"Overall zero fraction: {zero_fraction:.3f}")
print(f"Median per-sample zero fraction: {sample_zero_fraction.median():.3f}")
print(f"Median sequencing depth: {df_depth.median():.0f}")


Student CSV: microbiome_synthetic.csv -> shape: (150, 402)
Instructor key: instructor_answer_key.csv -> shape: (40, 4)

Class balance:
health_status
Case       59
Control    91
Name: count, dtype: int64

Class proportions:
health_status
Case       0.393
Control    0.607
Name: proportion, dtype: float64
Imbalance ratio majority/minority: 1.54

Batch balance:
batch
batch_1    60
batch_2    27
batch_3    63
Name: count, dtype: int64

Confounding check (status x batch):
batch          batch_1  batch_2  batch_3
health_status                           
Case                14       10       35
Control             46       17       28

Confounding check, row-normalized:
batch          batch_1  batch_2  batch_3
health_status                           
Case             0.237    0.169    0.593
Control          0.505    0.187    0.308

Sequencing depth by health_status:
               median_depth  mean_depth
health_status                          
Case                36145.0     38274.0
Control  